# Grid Up Datathon — Veri Analizi

Bu defter, modelleme kararlarini belirleyen olcumleri tek yerde toplar.
Amac betimleyici bir tarama degil; her hucre bir KARARA baglanir.

Sira:
1. Varyans ayristirmasi — problem nerede?
2. Trafo seviyesini ne belirliyor (cold tavani)
3. Hava gercekte ne kadar aciklayabilir
4. Hafta sonu / takvim: ortalama yaniltiyor
5. Sifirlar: satir olayi degil varlik durumu


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

from src import config as C
from src.external.weather import load_weather, weather_features

tr = pd.read_csv('../data/raw/train.csv', parse_dates=['tarih'], dtype={'tanim':'string'})
tr['l'] = np.log1p(tr.tuketim.clip(lower=0))
nz = tr[tr.tuketim > 0].copy()
print(f'{len(tr):,} satir, {tr.tanim.nunique():,} trafo, sifir orani {(tr.tuketim<=0).mean():.2%}')

## 1. Varyans ayristirmasi

Iki yonlu sabit etki: `log1p(y) = trafo etkisi + gun etkisi + etkilesim`.
Hangi bilesenin buyuk oldugu, muhendislik cabasinin nereye gitmesi
gerektigini soyler.

In [ ]:
gm = nz.l.mean()
tot = ((nz.l - gm)**2).sum()
em = nz.groupby('tanim').l.transform('mean')
dm = nz.groupby('tarih').l.transform('mean')
resid = nz.l - em - dm + gm

pay = {
    'trafolar arasi': ((em-gm)**2).sum()/tot,
    'ortak zaman (mevsim+hava)': ((dm-gm)**2).sum()/tot,
    'trafo x gun': (resid**2).sum()/tot,
}
for k, v in pay.items():
    print(f'{k:28s} {v:6.1%}')
print(f'
toplam varyans {nz.l.var():.4f}')

**Karar.** Ortak zaman etkisi %1.4. Butun hava/takvim muhendisligi bu
kutunun icinde. Oncelik trafo seviyesi tahmini (%92).

## 2. Cold trafo tavani

Cold trafonun gecmisi yok; seviyesini yalnizca statik nitelikler ima eder.

In [ ]:
ent = nz.groupby('tanim').agg(lvl=('l','mean'), guc=('guc','first'), lok=('lokasyon','first'))
v = ent.lvl.var()
def r2(keys):
    return 1 - ((ent.lvl - ent.groupby(keys).lvl.transform('mean'))**2).mean()/v
print(f'R2(guc)            = {r2("guc"):.3f}')
print(f'R2(lokasyon)       = {r2("lok"):.3f}')
print(f'R2(guc + lokasyon) = {r2(["guc","lok"]):.3f}')

erisilemez = (1 - r2(['guc','lok'])) * (pay['trafolar arasi'] * nz.l.var())
floor = np.sqrt(erisilemez + (resid**2).mean())
print(f'
cold RMSLE tavani ~ {floor:.2f}  (reports/02 bagimsiz olarak 1.09-1.14 buldu)')

## 3. Hava: tavan ve neden global degiskenler ise yaramiyor

In [ ]:
w = weather_features(load_weather())
cols = ['cdd','hdd','temperature_2m_mean','relative_humidity_2m_mean',
        'precipitation_sum','wind_speed_10m_max','sunshine_duration']
d = nz.merge(w[['lokasyon','tarih']+cols], on=['lokasyon','tarih'], how='inner')
d['r'] = d.l - d.groupby('tanim').l.transform('mean') - d.groupby('tarih').l.transform('mean') + d.l.mean()

cnt = d.groupby('tanim').size(); big = cnt[cnt >= 200].index
s = d[d.tanim.isin(big)]
tot_ss = (s.r**2).sum(); expl = 0.0
for f in ['cdd','hdd']:
    x = s[f] - s.groupby('tanim')[f].transform('mean')
    beta = ((x*s.r).groupby(s.tanim).transform('sum') /
            (x*x).groupby(s.tanim).transform('sum').replace(0, np.nan)).fillna(0)
    expl += ((beta*x)**2).sum()
print(f'trafo-bazli CDD/HDD, trafo x gun artiginin %{expl/tot_ss*100:.1f}'ini acikliyor (ORNEKLEM-ICI)')
print(f'toplam varyanstaki karsiligi: %{expl/tot_ss*pay["trafo x gun"]*100:.2f}')

In [ ]:
x = s['cdd'] - s.groupby('tanim')['cdd'].transform('mean')
beta = ((x*s.r).groupby(s.tanim).sum() / (x*x).groupby(s.tanim).sum().replace(0, np.nan)).dropna()
t, p = stats.ttest_1samp(beta, 0)
print(f'CDD egimi: ort {beta.mean():+.4f}  t={t:.1f}  p={p:.2f}  <- ANLAMSIZ')
print(f'ama std {beta.std():.4f}, %10-%90 [{beta.quantile(.1):+.3f}, {beta.quantile(.9):+.3f}]')
beta.hist(bins=60); plt.axvline(0, color='r'); plt.title('Trafo bazinda CDD egimi'); plt.show()

**Karar.** Ortalama trafonun sicaklik tepkisi sifirdan farksiz; tepkiler
trafo bazinda var ama isaretleri karisik. Global CDD kolonu bilgi tasimiyor,
yalnizca trafo-bazli katsayi tasiyabilir.

## 4. Hafta sonu: ortalama yaniltiyor

In [ ]:
nz['wknd'] = nz.tarih.dt.dayofweek >= 5
prof = nz.groupby(['tanim','wknd']).l.mean().unstack().dropna()
diff = prof[True] - prof[False]
t, p = stats.ttest_1samp(diff, 0)
print(f'ortalama {diff.mean():+.4f} (t={t:.1f}, p={p:.1e})  ama MEDYAN {diff.median():+.4f}')
print(f'hafta sonu ARTAN trafo orani: %{(diff>0).mean()*100:.0f}')
diff.clip(-1,1).hist(bins=80); plt.axvline(0, color='r'); plt.title('Hafta sonu - hafta ici'); plt.show()

## 5. Sifirlar: varlik durumu, satir olayi degil

In [ ]:
zr = tr.groupby('tanim').tuketim.apply(lambda s: (s<=0).mean())
print(f'hic sifiri olmayan trafo : {(zr==0).sum():,}')
print(f'sifir orani >0.9        : {(zr>0.9).sum():,}')
print(f'arada (0.1-0.9)         : {((zr>0.1)&(zr<=0.9)).sum():,}  <- bos orta bolge')
zr[zr>0].hist(bins=50); plt.title('Trafo bazinda sifir orani (>0)'); plt.show()

**Karar.** Sifir, gunluk bir olay degil kalici bir varlik durumu. Bu yuzden
tahmin "bugun sifir mi" degil "bu trafo olu mu" sorusuna baglanir ve
`(1-p) * seviye` seklinde kucultme uygulanir (`src/models/zero_gate.py`).